In [16]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

print(f"Data will be saved to: {DATA_RAW}")


Data will be saved to: d:\Workspace\Mon_Hoc\datamining\final-project\vn30-ver1\data\raw


In [17]:
from vnstock import Listing

listing = Listing(source="VCI")
vn30_symbols = listing.symbols_by_group("VN30")  # pandas Series 30 mã
vn30_symbols.tolist()


['ACB',
 'BCM',
 'BID',
 'CTG',
 'DGC',
 'FPT',
 'GAS',
 'GVR',
 'HDB',
 'HPG',
 'LPB',
 'MBB',
 'MSN',
 'MWG',
 'PLX',
 'SAB',
 'SHB',
 'SSB',
 'SSI',
 'STB',
 'TCB',
 'TPB',
 'VCB',
 'VHM',
 'VIB',
 'VIC',
 'VJC',
 'VNM',
 'VPB',
 'VRE']

In [18]:
import time
import pandas as pd
from vnstock import Quote

START_DATE = "2020-01-01"
END_DATE = "2025-11-30"

all_frames = []

for sym in vn30_symbols.tolist():
    print(f"Downloading {sym}...")
    quote = Quote(symbol=sym, source="VCI")
    df = quote.history(start=START_DATE, end=END_DATE)
    df["symbol"] = sym
    all_frames.append(df)
    time.sleep(0.2)

# Gộp tất cả dữ liệu
df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.rename(columns={"time": "date"})

print(f"\nTotal rows: {len(df_raw)}")
print(f"Symbols: {df_raw['symbol'].nunique()}")
df_raw.head()



Total rows: 45977
Symbols: 30


,date,open,high,low,close,volume,symbol
0,2019-09-19,6.46,6.70,6.44,6.67,3421555,ACB
1,2019-09-20,6.78,7.33,6.67,6.70,2535289,ACB
2,2019-09-23,6.70,6.81,6.67,6.72,2878182,ACB
3,2019-09-24,6.72,6.75,6.67,6.70,1324749,ACB
4,2019-09-25,6.67,6.70,6.61,6.64,1522720,ACB


In [ ]:
# Làm sạch và chuẩn hóa dữ liệu
df_clean = df_raw.copy()

# Chuyển đổi datetime
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean = df_clean.dropna(subset=["date"])

# Lọc theo khoảng thời gian
df_clean = df_clean[(df_clean["date"] >= START_DATE) & (df_clean["date"] <= END_DATE)]

# Sắp xếp theo symbol và date
df_clean = df_clean.sort_values(["symbol", "date"]).reset_index(drop=True)

# Kiểm tra
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Symbols: {df_clean['symbol'].nunique()}")
print(f"Total rows: {len(df_clean)}")

# Lưu file
output_path = DATA_RAW / "vn30_stock_data.csv"
df_clean.to_csv(output_path, index=False)
print("Saved to: {output_path}")


Date range: 2020-01-02 00:00:00 to 2025-11-28 00:00:00
Symbols: 30
Total rows: 43932

✓ Saved to: d:\Workspace\Mon_Hoc\datamining\final-project\vn30-ver1\data\raw\vn30_stock_data.csv
